# Qwen3.5-0.8B typed decisions / Microsoft Olive

Run the cells in order on Linux with an NVIDIA CUDA GPU that supports BF16 (for example, a suitable Colab runtime). The notebook builds [Microsoft Olive from a pinned source revision](https://github.com/microsoft/Olive), prepares the train and validation splits of [typed-decisions-synth](https://huggingface.co/datasets/n4ze3m/typed-decisions-synth), generates an Olive configuration, and trains in the notebook kernel. The question format and option shuffling follow [Hmm train.py](https://github.com/n4ze3m/hmm/blob/main/training/train.py); the workflow is based on the [Olive finetune CLI](https://microsoft.github.io/Olive/how-to/cli/cli-finetune.html).

**Important difference:** Olive's default LoRA fine-tuning uses full-sequence language-model SFT. It does not reproduce the reference script's candidate-letter-only loss mixed with teacher soft labels, nor does it reshuffle options at every step. Teacher probabilities are retained for analysis, but training uses only the gold letter. Do not assume that this adapter is calibrated like Hmm. Olive normally joins samples into a continuous corpus; this notebook explicitly switches to `line-by-line` after generating the configuration.

In [ ]:
# 1. Build an Olive wheel from source rather than installing a prebuilt PyPI package.
#    Install the LoRA extra and the CUDA kernels required by Qwen3.5 Gated DeltaNet.
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd() / "olive_typed_decisions"
SOURCE = ROOT / "Olive"
WHEELS = ROOT / "wheels"
OLIVE_REVISION = "2fbeaf4316930d62bf7b85658ccc6e752d4b6f4c"
PIP_INDEX = "https://packagefeedproxy.microsoft.io/pypi/simple/"
ROOT.mkdir(parents=True, exist_ok=True)
WHEELS.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    subprocess.run(["git", "clone", "https://github.com/microsoft/Olive.git", str(SOURCE)], check=True)
elif not (SOURCE / ".git").is_dir() or not (SOURCE / "setup.py").is_file():
    raise RuntimeError(f"Not an Olive source checkout: {SOURCE}")
subprocess.run(["git", "-C", str(SOURCE), "fetch", "origin", OLIVE_REVISION], check=True)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", OLIVE_REVISION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", PIP_INDEX, "build"], check=True)
for old_wheel in WHEELS.glob("olive_ai-*.whl"):
    old_wheel.unlink()
subprocess.run([sys.executable, "-m", "build", "--wheel", "--outdir", str(WHEELS), str(SOURCE)], check=True)
wheels = sorted(WHEELS.glob("olive_ai-*.whl"))
if len(wheels) != 1:
    raise RuntimeError(f"Expected exactly one Olive wheel; found {wheels}")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--index-url", PIP_INDEX, "--upgrade",
    f"{wheels[0]}[lora]", "datasets", "transformers>=4.57.0", "ninja",
    "flash-linear-attention[cuda]",
], check=True)
# Pin CUDA 12-compatible ONNX Runtime versions for the Colab A100 runtime.
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "onnxruntime", "onnxruntime-gpu", "onnxruntime-genai", "onnxruntime-genai-cuda",
], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--index-url", PIP_INDEX, "--upgrade",
    "onnxruntime-gpu==1.26.0", "onnxruntime-genai-cuda==0.14.0",
], check=True)
# Build causal-conv1d against the PyTorch/CUDA installation in the current kernel.
subprocess.run([
    sys.executable, "-m", "pip", "install", "--index-url", PIP_INDEX,
    "--upgrade", "--no-build-isolation", "causal-conv1d>=1.4.0",
], check=True)
# Dependency resolution may reinstall Colab's old torchao; BF16 LoRA does not need it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
from importlib import metadata
try:
    installed_torchao = metadata.version("torchao")
except metadata.PackageNotFoundError:
    installed_torchao = None
if installed_torchao is not None:
    raise RuntimeError(f"torchao {installed_torchao} is still installed; restart the runtime and rerun cell 1")
print("Built Olive from:", subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# 2. Check hardware and versions before starting the CUDA LoRA workflow (not CPU/MPS).
import onnxruntime as ort
import onnxruntime_genai as og
import torch
import transformers
from importlib import metadata
from datasets import load_dataset
from transformers import AutoTokenizer

try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    raise RuntimeError(f"Unexpected torchao {torchao_version}; rerun cell 1, then restart the runtime")
ort_version = metadata.version("onnxruntime-gpu")
ort_genai_version = metadata.version("onnxruntime-genai-cuda")
if (ort_version, ort_genai_version) != ("1.26.0", "0.14.0"):
    raise RuntimeError(
        f"Expected onnxruntime-gpu 1.26.0 + onnxruntime-genai-cuda 0.14.0; found "
        f"{ort_version} + {ort_genai_version}. Rerun cell 1, restart the runtime, then rerun from cell 2."
    )
if "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError(
        f"ONNX Runtime CUDA provider failed to load; available providers: {ort.get_available_providers()}. "
        "Rerun cell 1 and restart the runtime before continuing."
    )
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("This recipe requires a CUDA GPU with bfloat16 support; use a suitable Linux GPU runtime.")
try:
    from causal_conv1d import causal_conv1d_fn
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule
except ImportError as exc:
    raise RuntimeError("Qwen3.5 CUDA kernels are unavailable; rerun cell 1 and restart the runtime") from exc
MODEL_ID = "Qwen/Qwen3.5-0.8B"
MAX_SEQ_LEN = 768
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
letter_ids = [tokenizer.encode(letter, add_special_tokens=False) for letter in LETTERS]
if any(len(ids) != 1 for ids in letter_ids):
    raise ValueError("Each option letter must be one tokenizer token; adapt the prompt before training.")
print(
    "transformers:", transformers.__version__,
    "CUDA:", torch.cuda.get_device_name(0),
    "ORT:", ort.__version__,
    "ORT GenAI:", ort_genai_version,
    "Qwen3.5 kernels: enabled",
)


## Data preparation

The source dataset stores `questions`, `gold`, and `teacher` as JSON **strings**. Preserve its case-level train/validation split and create one sample per question. Shuffle choice options deterministically using the state and question ID, keep score levels in order, and use false/true for noul. Discard overlong samples rather than truncating away their answers. The `text` field in the prepared JSONL is used for Olive SFT; the other fields support first-token candidate-probability checks.

In [ ]:
# 3. Format option-letter prompts like train.py; retain teacher soft labels for analysis only.
import hashlib
import json
import random

def as_text(value):
    return value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)

def make_examples(case):
    state = as_text(json.loads(case["state"]) if case["state_is_json"] else case["state"])
    questions = json.loads(case["questions"])
    gold = json.loads(case["gold"])
    teachers = json.loads(case["teacher"]) if case.get("teacher") else {}
    for qid, question in questions.items():
        answer = gold[qid]
        teacher = teachers.get(qid)
        kind = question["type"]
        if kind == "choice":
            options = [(str(k), as_text(v)) for k, v in question["criteria"].items()]
            seed = int.from_bytes(hashlib.sha256(f"{state}\0{qid}".encode()).digest()[:8], "big")
            random.Random(seed).shuffle(options)
            answer_key = str(answer)
            probs = teacher["probabilities"] if teacher else None
        elif kind == "score":
            options = [(str(i), as_text(v)) for i, v in enumerate(question["criteria"])]
            answer_key = str(answer)
            probs = teacher["probabilities"] if teacher else None
        elif kind == "noul":
            criteria = question.get("criteria") or {}
            options = [("false", as_text(criteria.get("false", "No"))), ("true", as_text(criteria.get("true", "Yes")))]
            answer_key = "true" if answer in (True, "true", "yes") else "false"
            probs = {"false": 1 - teacher["noul"], "true": teacher["noul"]} if teacher else None
        else:
            raise ValueError(f"Unsupported question type: {kind}")
        keys = [key for key, _ in options]
        if not 2 <= len(keys) <= len(LETTERS) or answer_key not in keys:
            raise ValueError(f"Invalid options/answer for case {case['state_id']}, question {qid}")
        label = keys.index(answer_key)
        prompt_text = (
            "State (data to evaluate):\n" + state + "\n\nQuestion:\n" + as_text(question["instructions"])
            + "\n\nOptions:\n" + "\n".join(f"{LETTERS[i]}: {key} — {description}" for i, (key, description) in enumerate(options))
            + "\nReturn only the option letter."
        )
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}], tokenize=False,
            add_generation_prompt=True, enable_thinking=False,
        )
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}, {"role": "assistant", "content": LETTERS[label]}],
            tokenize=False, enable_thinking=False,
        )
        if not text.startswith(prompt):
            raise ValueError("Qwen chat template changed: the training prefix differs from the inference prompt.")
        if len(tokenizer.encode(text, add_special_tokens=False)) > MAX_SEQ_LEN:
            continue
        soft = None
        if probs is not None:
            soft = [max(0.0, float(probs.get(key, 0))) for key in keys]
            total = sum(soft)
            soft = [p / total for p in soft] if total > 0 else None
        yield {"text": text, "prompt": prompt, "label": label, "n_options": len(keys), "soft": soft, "type": kind}


In [ ]:
# 4. Preserve the case-level split so the same state cannot leak into validation.
dataset = load_dataset("n4ze3m/typed-decisions-synth")
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
counts = {}
for split in ("train", "validation"):
    if split not in dataset:
        raise KeyError(f"Dataset is missing the {split} split")
    count = 0
    with (DATA_DIR / f"{split}.jsonl").open("w", encoding="utf-8") as stream:
        for case in dataset[split]:
            for sample in make_examples(case):
                stream.write(json.dumps(sample, ensure_ascii=False) + "\n")
                count += 1
    if not count:
        raise RuntimeError(f"No usable examples in {split}")
    counts[split] = count
print("Prepared questions after length filtering:", counts)


## Olive CLI finetune

Use `olive finetune --dry_run` from the **source-built** Olive installation to generate a configuration for the installed version. Then set the `line-by-line` strategy and separate train/validation file mappings, which the CLI cannot express directly. The dry run only writes a configuration; training is started later in the notebook kernel. Use a small per-device batch to limit GPU memory usage. Hyperparameters are inspired by Hmm: LoRA rank 32, alpha 64, learning rate 1e-4, and one epoch.

In [ ]:
# 5. Generate a version-matched template with the Olive CLI rather than hand-writing its schema.
import shutil

olive = str(Path(sys.executable).parent / "olive") if (Path(sys.executable).parent / "olive").is_file() else shutil.which("olive")
if olive is None:
    raise RuntimeError("olive CLI is not installed in this notebook kernel's environment")
OUTPUT = ROOT / "finetuned"
TRAINER = ROOT / "trainer_checkpoints"
TARGETS = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"
subprocess.run([
    olive, "finetune", "--model_name_or_path", MODEL_ID,
    "--output_path", str(OUTPUT), "--data_name", "json",
    "--data_files", str(DATA_DIR / "train.jsonl"), "--text_field", "text",
    "--method", "lora", "--torch_dtype", "bfloat16",
    "--target_modules", TARGETS, "--lora_r", "32", "--lora_alpha", "64",
    "--max_seq_len", str(MAX_SEQ_LEN), "--max_samples", str(counts["train"]),
    "--learning_rate", "1e-4", "--num_train_epochs", "1",
    "--per_device_train_batch_size", "1", "--gradient_accumulation_steps", "16",
    "--output_dir", str(TRAINER), "--logging_steps", "50", "--save_strategy", "no",
    "--log_level", "1", "--dry_run",
], check=True)
config_path = OUTPUT / "config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
print("Olive generated:", config_path)


In [ ]:
# 6. Replace JOIN with one sequence per question, remove the preprocessing cap, and add validation.
from copy import deepcopy

train_data = config["data_configs"][0]
train_data["load_dataset_config"]["data_files"] = {
    "train": str(DATA_DIR / "train.jsonl"),
    "validation": str(DATA_DIR / "validation.jsonl"),
}
train_data["load_dataset_config"]["split"] = "train"
train_data["pre_process_data_config"].update({
    "strategy": "line-by-line", "pad_to_max_len": False, "max_samples": None,
})
eval_data = deepcopy(train_data)
eval_data["name"] = "eval_data"
eval_data["load_dataset_config"]["split"] = "validation"
config["data_configs"].append(eval_data)
config["passes"]["f"]["eval_data_config"] = "eval_data"
config["passes"]["f"]["training_args"].update({
    "eval_strategy": "epoch",
    "overwrite_output_dir": True,
    "report_to": [],
})
config["clean_cache"] = True

# Validate the Olive workflow schema before loading the model or allocating GPU memory.
from olive.workflows.run.config import RunConfig
RunConfig.model_validate(deepcopy(config))
config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Validated line-by-line train/validation config:", counts)


In [ ]:
# 7. Train only when this cell runs; preceding cells do not update adapter weights.
#    Call Olive in this notebook kernel to avoid a mismatched /usr/local/bin/olive environment.
from olive.workflows import run as olive_run

workflow_output = olive_run(config)
if not workflow_output.has_output_model():
    raise RuntimeError("Olive completed without an output model; inspect the traceback/log above.")
adapter_dir = OUTPUT / "adapter"
if not (adapter_dir / "adapter_config.json").is_file():
    raise RuntimeError(f"Olive finished but no PEFT adapter found at {adapter_dir}; inspect {OUTPUT}")
print("PEFT adapter:", adapter_dir)


In [ ]:
OUTPUT = Path("/content/H100/olive_typed_decisions/finetuned")

## Export to ONNX Runtime GenAI

Use Olive `ModelBuilder` to export the base model with the trained PEFT adapter as a CUDA FP16 ONNX model. The notebook pins CUDA 12-compatible `onnxruntime-genai-cuda==0.14.0` and `onnxruntime-gpu==1.26.0`; restart the Colab runtime after changing native runtime packages. The output directory contains `model.onnx`, external weights, tokenizer files, and `genai_config.json`.

In [ ]:
# 8. Export the base model and fine-tuned adapter with Olive ModelBuilder to CUDA FP16 ONNX.
import json
import shutil
from copy import deepcopy
from importlib import metadata
from pathlib import Path

from huggingface_hub import snapshot_download
from olive.passes.onnx.model_builder import ModelBuilder as OliveModelBuilder
from olive.workflows import run as olive_run
from olive.workflows.run.config import RunConfig
from onnxruntime_genai.models import builder as ort_genai_builder

ROOT = Path.cwd() / "olive_typed_decisions"
OUTPUT = ROOT / "finetuned"
MODEL_ID = "Qwen/Qwen3.5-0.8B"
adapter_dir = OUTPUT / "adapter"
if not (adapter_dir / "adapter_config.json").is_file():
    raise FileNotFoundError(f"Run cell 7 first; no PEFT adapter found at {adapter_dir}")
BASE_MODEL_DIR = ROOT / "hf_base_model"
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=BASE_MODEL_DIR,
    token=False,
)
if not list(BASE_MODEL_DIR.glob("*.safetensors")):
    raise RuntimeError(f"No safetensors weights downloaded to {BASE_MODEL_DIR}")
# ORT GenAI Builder reads eos_token_id at the top level; Qwen3.5 stores it in text_config.
base_config_path = BASE_MODEL_DIR / "config.json"
base_config = json.loads(base_config_path.read_text(encoding="utf-8"))
if base_config.get("eos_token_id") is None:
    eos_token_id = (base_config.get("text_config") or {}).get("eos_token_id")
    if eos_token_id is None:
        raise RuntimeError(f"No eos_token_id found in {base_config_path}")
    base_config["eos_token_id"] = eos_token_id
    base_config_path.write_text(json.dumps(base_config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
ONNX_OUTPUT = ROOT / "onnx_fp16_cuda"
# Do not reuse artifacts generated by a different ORT GenAI version.
if ONNX_OUTPUT.exists():
    shutil.rmtree(ONNX_OUTPUT)
# This Olive revision targets newer ORT GenAI APIs, including quantization loaders
# and a changed check_extra_options signature; v0.14 FP16 export needs no quantization patch.
olive_builder_originals = None
if metadata.version("onnxruntime-genai-cuda") == "0.14.0":
    def _skip_quant_patch_for_fp16():
        return None

    def _check_v014_extra_options(
        _model_name, _input_path, _output_dir, _precision, execution_provider, extra_options
    ):
        for key, value in list(extra_options.items()):
            if isinstance(value, bool):
                extra_options[key] = str(value).lower()
        ort_genai_builder.check_extra_options(extra_options, execution_provider)

    olive_builder_originals = (
        OliveModelBuilder.__dict__["maybe_patch_quant"],
        OliveModelBuilder.__dict__["_check_extra_options"],
    )
    OliveModelBuilder.maybe_patch_quant = staticmethod(_skip_quant_patch_for_fp16)
    OliveModelBuilder._check_extra_options = staticmethod(_check_v014_extra_options)
onnx_config = {
    "input_model": {
        "type": "HfModel",
        # Qwen3.5 MTP weight loading requires local safetensors, not only a Hub repo ID.
        "model_path": str(BASE_MODEL_DIR),
        "adapter_path": str(adapter_dir),
    },
    "systems": {
        "local_system": {
            "type": "LocalSystem",
            "accelerators": [{"device": "gpu", "execution_providers": ["CUDAExecutionProvider"]}],
        }
    },
    "passes": {
        "model_builder": {
            "type": "ModelBuilder",
            "precision": "fp16",
            # v0.14 supports Qwen3.5 text-only export; fetch this public model anonymously.
            "extra_options": {
                "filename": "model.onnx",
                "hf_token": "false",
                # Keep token embeddings for standalone text tests instead of exporting only inputs_embeds.
                "exclude_embeds": False,
            },
        }
    },
    "host": "local_system",
    "target": "local_system",
    "output_dir": str(ONNX_OUTPUT),
    "clean_cache": True,
    "log_severity_level": 0,
    "log_to_file": True,
    "no_artifacts": True,
}
RunConfig.model_validate(deepcopy(onnx_config))
logs_before = set(Path.cwd().glob("olive-*.log"))
try:
    onnx_workflow_output = olive_run(onnx_config)
finally:
    if olive_builder_originals is not None:
        OliveModelBuilder.maybe_patch_quant, OliveModelBuilder._check_extra_options = olive_builder_originals
if not onnx_workflow_output.has_output_model():
    log_candidates = sorted(set(Path.cwd().glob("olive-*.log")) - logs_before, key=lambda path: path.stat().st_mtime)
    if not log_candidates:
        log_candidates = sorted(Path.cwd().glob("olive-*.log"), key=lambda path: path.stat().st_mtime)
    log_tail = ""
    if log_candidates:
        log_path = log_candidates[-1]
        log_tail = "\nLog: " + str(log_path) + "\n" + "".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-120:])
    raise RuntimeError("Olive ONNX conversion completed without an output model." + log_tail)
genai_configs = list(ONNX_OUTPUT.rglob("genai_config.json"))
if len(genai_configs) != 1:
    raise RuntimeError(f"Expected one genai_config.json under {ONNX_OUTPUT}; found {genai_configs}")
onnx_model_dir = genai_configs[0].parent
onnx_files = list(onnx_model_dir.glob("*.onnx"))
if not onnx_files:
    raise RuntimeError(f"No ONNX model was generated in {onnx_model_dir}")
print("ONNX Runtime GenAI model:", onnx_model_dir)
print("ONNX files:", [path.name for path in onnx_files])


In [ ]:
# 9. Generate a standalone test script that can check predicted letters against validation.jsonl.
TEST_SCRIPT = '''import argparse
import json
import re
from pathlib import Path

import numpy as np
import onnxruntime_genai as og

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def load_sample(path: Path, index: int) -> dict:
    with path.open(encoding="utf-8") as stream:
        for current, line in enumerate(stream):
            if current == index:
                return json.loads(line)
    raise IndexError(f"Sample index {index} is outside {path}")


def generate(model_path: Path, prompt: str, max_new_tokens: int) -> str:
    model = og.Model(str(model_path))
    tokenizer = og.Tokenizer(model)
    input_ids = tokenizer.encode(prompt)
    params = og.GeneratorParams(model)
    params.set_search_options(max_length=len(input_ids) + max_new_tokens, do_sample=False)
    generator = og.Generator(model, params)
    generator.append_tokens(input_ids)
    while not generator.is_done():
        generator.generate_next_token()
    sequence = np.asarray(generator.get_sequence(0), dtype=np.int32)
    return tokenizer.decode(sequence[len(input_ids):]).strip()


def main() -> None:
    parser = argparse.ArgumentParser(description="Test the Olive-exported Qwen3.5 ONNX model.")
    parser.add_argument("--model", type=Path, required=True, help="Directory containing genai_config.json.")
    parser.add_argument("--data", type=Path, help="Prepared validation JSONL containing prompt and label.")
    parser.add_argument("--index", type=int, default=0, help="Validation sample index.")
    parser.add_argument("--prompt", help="Already chat-templated prompt; skips gold-label checking.")
    parser.add_argument("--max-new-tokens", type=int, default=8)
    parser.add_argument("--check", action="store_true", help="Exit non-zero when prediction differs from gold.")
    args = parser.parse_args()
    if not (args.model / "genai_config.json").is_file():
        raise FileNotFoundError(f"Missing genai_config.json in {args.model}")
    if args.prompt is None and args.data is None:
        parser.error("provide --prompt or --data")
    expected = None
    n_options = len(LETTERS)
    prompt = args.prompt
    if prompt is None:
        sample = load_sample(args.data, args.index)
        prompt = sample["prompt"]
        expected = LETTERS[sample["label"]]
        n_options = sample["n_options"]
    output = generate(args.model, prompt, args.max_new_tokens)
    match = re.search(rf"\\b([{LETTERS[:n_options]}])\\b", output.upper())
    prediction = match.group(1) if match else None
    result = {"output": output, "prediction": prediction, "expected": expected, "correct": expected is None or prediction == expected}
    print(json.dumps(result, ensure_ascii=False, indent=2))
    if args.check and expected is not None and prediction != expected:
        raise AssertionError(f"Prediction {prediction!r} does not match expected {expected!r}")


if __name__ == "__main__":
    main()
'''
test_script_path = ROOT / "test_qwen35_onnx.py"
test_script_path.write_text(TEST_SCRIPT, encoding="utf-8")
print("Test script:", test_script_path)
print(f"Run: {sys.executable} {test_script_path} --model {onnx_model_dir} --data {DATA_DIR / 'validation.jsonl'} --index 0")


In [ ]:
DATA_DIR

In [ ]:
# 10. Test one validation sample against the exported model, tokenizer, and generation loop.
import runpy
import sys

previous_argv = sys.argv
try:
    sys.argv = [
        str(test_script_path),
        "--model", str(onnx_model_dir),
        "--data", str(DATA_DIR / "validation.jsonl"),
        "--index", "1",
    ]
    runpy.run_path(str(test_script_path), run_name="__main__")
finally:
    sys.argv = previous_argv


In [ ]:
# 11. Check noul and choice using first-token ONNX logits, following the Hmm model card.
import json

import numpy as np
import onnxruntime_genai as og

HMM_REQUEST = {
    "state": "Help! My payouts have failed for 3 days. I need the money today.",
    "questions": {
        "is_urgent": {
            "type": "noul",
            "instructions": "Does this message convey urgency?",
        },
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payments, invoicing, refunds",
                "technical": "Bugs, outages, integrations",
                "sales": "Pricing, upgrades, new accounts",
            },
        },
    },
}


def hmm_text(value):
    return value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def hmm_options(question):
    kind = question["type"]
    criteria = question.get("criteria") or {}
    if kind == "choice":
        return [(str(key), hmm_text(value)) for key, value in criteria.items()]
    if kind == "score":
        return [(str(index), hmm_text(value)) for index, value in enumerate(criteria)]
    if kind == "noul":
        return [("false", hmm_text(criteria.get("false", "No"))), ("true", hmm_text(criteria.get("true", "Yes")))]
    raise ValueError(f"Unsupported question type: {kind}")


def hmm_prompt(state, question, options):
    option_lines = "\n".join(
        f"{LETTERS[index]}: {key} — {description}"
        for index, (key, description) in enumerate(options)
    )
    user = (
        "State (data to evaluate):\n" + hmm_text(state)
        + "\n\nQuestion:\n" + hmm_text(question["instructions"])
        + "\n\nOptions:\n" + option_lines
        + "\nReturn only the option letter."
    )
    return f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"


onnx_hmm_model = og.Model(str(onnx_model_dir))
onnx_hmm_tokenizer = og.Tokenizer(onnx_hmm_model)
letter_token_ids = []
for letter in LETTERS:
    ids = np.asarray(onnx_hmm_tokenizer.encode(letter), dtype=np.int32).reshape(-1)
    if ids.size != 1:
        raise ValueError(f"Option letter {letter!r} is not one ONNX tokenizer token: {ids.tolist()}")
    letter_token_ids.append(int(ids[0]))


def onnx_first_token_probabilities(prompt, n_options):
    input_ids = np.asarray(onnx_hmm_tokenizer.encode(prompt), dtype=np.int32).reshape(-1)
    params = og.GeneratorParams(onnx_hmm_model)
    params.set_search_options(max_length=len(input_ids) + 1, do_sample=False)
    generator = og.Generator(onnx_hmm_model, params)
    generator.append_tokens(input_ids)
    logits = np.asarray(generator.get_logits(), dtype=np.float32)
    next_token_logits = logits.reshape(-1, logits.shape[-1])[-1]
    candidate_logits = next_token_logits[letter_token_ids[:n_options]]
    candidate_logits -= candidate_logits.max()
    probabilities = np.exp(candidate_logits)
    probabilities /= probabilities.sum()
    del generator
    return probabilities, len(input_ids)


answers = {}
total_input_tokens = 0
for name, question in HMM_REQUEST["questions"].items():
    options = hmm_options(question)
    if not 2 <= len(options) <= len(LETTERS):
        raise ValueError(f"Question {name!r} must have 2-{len(LETTERS)} options")
    probabilities, input_tokens = onnx_first_token_probabilities(
        hmm_prompt(HMM_REQUEST["state"], question, options), len(options)
    )
    total_input_tokens += input_tokens
    keys = [key for key, _ in options]
    best = int(probabilities.argmax())
    rounded = {key: round(float(probabilities[index]), 4) for index, key in enumerate(keys)}
    confidence = round(float((len(options) * probabilities[best] - 1) / (len(options) - 1)), 4)
    if question["type"] == "noul":
        answers[name] = {"type": "noul", "noul": round(float(probabilities[1]), 4)}
    elif question["type"] == "choice":
        answers[name] = {
            "type": "choice",
            "choice": keys[best],
            "probabilities": rounded,
            "confidence": confidence,
        }
    else:
        answers[name] = {
            "type": "score",
            "score": round(float(sum(index * p for index, p in enumerate(probabilities))), 4),
            "legend": dict(options),
            "probabilities": rounded,
            "confidence": confidence,
        }

print(json.dumps({
    "model": str(onnx_model_dir),
    "answers": answers,
    "usage": {"input_tokens": total_input_tokens, "output_tokens": 0},
}, ensure_ascii=False, indent=2))


## Interpreting results

Olive's validation `eval_loss` is the **full-sequence** language-model loss; it is not Hmm's candidate-letter NLL, accuracy, or ECE. The optional cell below computes first-token metrics on the original validation split. At inference time, use the same Qwen tokenizer and chat template with `add_generation_prompt=True, enable_thinking=False`, select the first-token logits for the first `n_options` letters, and apply softmax over those candidates to obtain choice/score/noul probabilities. Do not interpret unrestricted generated text as probabilities. Teacher probabilities are retained in the prepared data but are not used in Olive training.

In [ ]:
# 12. Optional: calculate first-token candidate accuracy, NLL, and 10-bin ECE as in Hmm.
#     These differ from Olive eval_loss; run this cell manually after training.
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16).cuda()
model = PeftModel.from_pretrained(base, adapter_dir).eval()
candidate_ids = [ids[0] for ids in letter_ids]
confidences, hits, nll = [], [], 0.0
with torch.inference_mode(), (DATA_DIR / "validation.jsonl").open(encoding="utf-8") as stream:
    for line in stream:
        sample = json.loads(line)
        inputs = tokenizer(sample["prompt"], return_tensors="pt", add_special_tokens=False).to("cuda")
        logits = model(**inputs, logits_to_keep=1, use_cache=False).logits[0, -1, candidate_ids[:sample["n_options"]]].float()
        probs = logits.softmax(-1)
        confidences.append(probs.max().item())
        hits.append(int(probs.argmax().item() == sample["label"]))
        nll += F.cross_entropy(logits[None, :], torch.tensor([sample["label"]], device="cuda")).item()
if not hits:
    raise RuntimeError("Validation JSONL contains no examples")
n = len(hits)
ece = 0.0
for bucket in range(10):
    indices = [i for i, c in enumerate(confidences) if bucket / 10 < c <= (bucket + 1) / 10]
    if indices:
        ece += len(indices) / n * abs(sum(hits[i] for i in indices) / len(indices) - sum(confidences[i] for i in indices) / len(indices))
print({"n": n, "accuracy": sum(hits) / n, "nll": nll / n, "ece": ece})


In [ ]:
# 13. Create a model card and upload the Olive-exported ONNX Runtime GenAI model to Hugging Face.
#     Use the login prompt with a write-enabled token for lokinfey; never store it in the notebook.
import os
from pathlib import Path

from huggingface_hub import HfApi, login

HF_REPO_ID = "lokinfey/Qwen3_5_0.8B_jev"
HF_PRIVATE = False

login(add_to_git_credential=False)

onnx_model_dir = Path(onnx_model_dir)
required_files = [onnx_model_dir / "genai_config.json", *onnx_model_dir.glob("*.onnx")]
if not required_files[0].is_file() or not any(path.is_file() for path in required_files[1:]):
    raise FileNotFoundError(f"Incomplete ONNX Runtime GenAI model at {onnx_model_dir}; run cell 8 first")

model_card = '''---
base_model: Qwen/Qwen3.5-0.8B
base_model_relation: finetune
datasets:
- n4ze3m/typed-decisions-synth
language:
- en
license: apache-2.0
library_name: onnxruntime
pipeline_tag: text-classification
tags:
- onnx
- onnxruntime-genai
- microsoft-olive
- peft
- lora
- qwen3.5
- typed-decisions
---

# Qwen3.5-0.8B Typed Decisions - ONNX Runtime GenAI

This repository contains a CUDA FP16 ONNX Runtime GenAI export of
[Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B), fine-tuned with a LoRA adapter
using [Microsoft Olive](https://github.com/microsoft/Olive) on
[n4ze3m/typed-decisions-synth](https://huggingface.co/datasets/n4ze3m/typed-decisions-synth).
The adapter is merged into the exported ONNX weights.

## Intended task

The model receives a state plus a typed `noul`, `choice`, or `score` question. It predicts an option
letter (`A`, `B`, `C`, ...) at the first generated position. Applications should select the candidate
letter logits and normalize only over the valid options instead of parsing unrestricted generated text.

## Runtime

The model was exported as FP16 for the CUDA execution provider with:

- `onnxruntime-genai-cuda==0.14.0`
- `onnxruntime-gpu==1.26.0`
- Microsoft Olive `2fbeaf4316930d62bf7b85658ccc6e752d4b6f4c`

These package versions target CUDA 12 environments. Load the directory containing `genai_config.json`.

```python
import numpy as np
import onnxruntime_genai as og
from huggingface_hub import snapshot_download

model_dir = snapshot_download("lokinfey/Qwen3_5_0.8B_jev")
model = og.Model(model_dir)
tokenizer = og.Tokenizer(model)

prompt = """<|im_start|>user
State (data to evaluate):
Help! My payouts have failed for 3 days. I need the money today.

Question:
Does this message convey urgency?

Options:
A: false — No
B: true — Yes
Return only the option letter.<|im_end|>
<|im_start|>assistant
<think>

</think>

"""

input_ids = np.asarray(tokenizer.encode(prompt), dtype=np.int32)
params = og.GeneratorParams(model)
params.set_search_options(max_length=len(input_ids) + 1, do_sample=False)
generator = og.Generator(model, params)
generator.append_tokens(input_ids)
raw_logits = np.asarray(generator.get_logits(), dtype=np.float32)
logits = raw_logits.reshape(-1, raw_logits.shape[-1])[-1]
letter_ids = [int(np.asarray(tokenizer.encode(letter)).reshape(-1)[0]) for letter in "AB"]
candidate_logits = logits[letter_ids]
probabilities = np.exp(candidate_logits - candidate_logits.max())
probabilities /= probabilities.sum()
print({"false": float(probabilities[0]), "true": float(probabilities[1])})
```

## Training

- Base model: `Qwen/Qwen3.5-0.8B`
- Dataset: `n4ze3m/typed-decisions-synth`, preserving its case-level train/validation split
- Sequence length: 768 tokens
- Method: LoRA, rank 32, alpha 64
- Target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- Epochs: 1
- Learning rate: 1e-4
- Precision: BF16 training, FP16 ONNX export

This Olive workflow uses standard full-sequence causal language-model SFT on the gold option letter. It does
**not** reproduce Hmm's candidate-only hard/teacher soft-label objective, so its probability calibration and
benchmark results should not be assumed to match `n4ze3m/Qwen3.5-4B-Hmm`.

## Evaluation

No benchmark result is published in this card. Evaluate accuracy, negative log-likelihood, and calibration on
your own held-out data by reading the first-token candidate logits. Olive's `eval_loss` is a full-sequence
language-model loss and is not directly comparable to candidate-letter NLL or accuracy.

## Limitations

- Intended for English prompts up to the 768-token training length.
- Supports at most 26 direct option letters.
- The synthetic training data and labels can contain errors or biases.
- Do not use this model as the sole safety, financial, legal, medical, or access-control decision maker.
- FP16 CUDA deployment requires compatible NVIDIA hardware and ONNX Runtime libraries.

## Attribution

Prompt structure and typed-decision evaluation follow the public
[Hmm project](https://github.com/n4ze3m/hmm). Model conversion and deployment use Microsoft Olive and
ONNX Runtime GenAI.
'''

(onnx_model_dir / "README.md").write_text(model_card, encoding="utf-8")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
api = HfApi()
account_name = api.whoami()["name"]
repo_owner = HF_REPO_ID.split("/", 1)[0]
if account_name != repo_owner:
    raise PermissionError(f"Logged-in Hugging Face account is {account_name!r}, not {repo_owner!r}; log in again with the correct account.")
if not api.repo_exists(repo_id=HF_REPO_ID, repo_type="model"):
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=HF_PRIVATE)
commit = api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(onnx_model_dir),
    path_in_repo=".",
    commit_message="Upload Olive FP16 ONNX Runtime GenAI model and Model Card",
    ignore_patterns=["*.log", "**/.ipynb_checkpoints/**"],
)
uploaded_files = {sibling.rfilename for sibling in api.model_info(HF_REPO_ID, files_metadata=False).siblings}
missing = {"README.md", "genai_config.json"} - uploaded_files
if missing or not any(name.endswith(".onnx") for name in uploaded_files):
    raise RuntimeError(f"Upload verification failed; missing={sorted(missing)}, files={sorted(uploaded_files)}")
print("Uploaded and verified:", commit)
print("Model page:", f"https://huggingface.co/{HF_REPO_ID}")
